In [1]:
#pip install mysql-connector-python
import mysql.connector
import pandas as pd
import numpy as np

In [2]:
# user = "admin", password = "Soybean3.0", dbname = "pat_database", host = "pat-database.ch7cvaspmesf.us-east-2.rds.amazonaws.com", port = 3306)
# full_ds <- dbReadTable(conn = pat, name = "raw_data_testing_updated"

In [2]:

# Replace with your actual database credentials
db_config = {
    'user': 'admin',
    'password': 'Soybean3.0',
    'host': 'pat-database.ch7cvaspmesf.us-east-2.rds.amazonaws.com',
    'database': 'pat_database'
}

try:
    # Establish a connection to the database
    conn = mysql.connector.connect(**db_config)

    # Create a cursor object
    cursor = conn.cursor()

    # Define your query
    query = '''
    
    SELECT CROP_SEASON, SEASON_NO, MACRO_REGION, Country, TRIAL_NAME, REP_NO, LOCATION_NAME, Crop_Variety, GRAIN_YLD_13PCT, SITE_ALT,
           TRIAL_STATUS, Org_Partner, Org_Source, PROTEIN, OIL, R6_RUST_Sev, R8_PL_Height, WGT_100_SEEDS, MAT_DAYS, PL_Lodging, Pod_Shattering 
    
    FROM raw_data_testing_updated
    WHERE TRIAL_STATUS = 'READY_FOR_ANALYSIS'

'''

     # Execute the query
    cursor.execute(query) 


    # Fetch all the results
    results = cursor.fetchall()

    # Get column names
    columns = [col[0] for col in cursor.description]

    # Create a DataFrame
    df = pd.DataFrame(results, columns=columns)


except mysql.connector.Error as err:
    print(f"Error: {err}")

finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()


In [3]:
# Create a DataFrame
df = pd.DataFrame(results, columns=columns)

# Columns to replace 0 with NaN
columns_to_replace = [
    'GRAIN_YLD_13PCT', 'TRIAL_STATUS', 'Org_Partner', 'Org_Source', 
    'PROTEIN', 'OIL', 'R6_RUST_Sev', 'R8_PL_Height', 
    'WGT_100_SEEDS', 'MAT_DAYS', 'PL_Lodging', 'Pod_Shattering'
]

# Replace 0 with NaN in specified columns
df[columns_to_replace] = df[columns_to_replace].replace(0, np.nan)

# Convert the 'MAT_DAYS' column to float
df['GRAIN_YLD_13PCT'] = df['GRAIN_YLD_13PCT'].astype(float)

In [4]:
# Convert the relevant columns to float
columns_to_convert = ['GRAIN_YLD_13PCT', 'MAT_DAYS', 'OIL', 'PROTEIN']
df[columns_to_convert] = df[columns_to_convert].astype(float)

# Define a function to find outliers in the specified columns
def find_outliers(group, columns):
    outlier_indices = []
    for column in columns:
        mean = group[column].mean()
        std = group[column].std()
        lower_bound = mean - 2 * std
        upper_bound = mean + 2 * std
        outliers = group[(group[column] < lower_bound) | (group[column] > upper_bound)].index
        outlier_indices.extend(outliers)
    
    # Remove duplicates in indices
    outlier_indices = list(set(outlier_indices))
    return group.loc[outlier_indices]

# Group by location and find outliers
outliers_df = df.groupby('LOCATION_NAME').apply(lambda x: find_outliers(x, columns_to_convert)).reset_index(drop=True)

In [6]:
columns_to_convert = ['GRAIN_YLD_13PCT', 'MAT_DAYS', 'OIL', 'PROTEIN']
df[columns_to_convert] = df[columns_to_convert].astype(float)

# Define a function to find outliers in the specified columns and replace them with NaN
def replace_outliers_with_nan(group, columns):
    for column in columns:
        mean = group[column].mean()
        std = group[column].std()
        lower_bound = mean - 2 * std
        upper_bound = mean + 2 * std
        outlier_condition = (group[column] < lower_bound) | (group[column] > upper_bound)
        group.loc[outlier_condition, column] = np.nan
    return group

# Group by location and apply the function
final_df = df.groupby('LOCATION_NAME').apply(lambda x: replace_outliers_with_nan(x, columns_to_convert)).reset_index(drop=True)


In [7]:
complete_dataset = {
    'CROP_SEASON': [],
    'MACRO_REGION': [],
    'Country': [],
    'TRIAL_NAME': [],
    'REP_NO': [],
    'Crop_Variety_head': [],
    'Crop_Variety_other': [],
    'GRAIN_YLD_13PCT_head': [],
    'GRAIN_YLD_13PCT_other': [],
    'Difference': [],
    'Win': [],
    'Lose': [],
    'Draw':[]
}

# Iterate through unique combinations of TRIAL_NAME and REP_NO
for (trial_name, rep_no), group in final_df.groupby(['TRIAL_NAME', 'REP_NO']):
    unique_crop_varieties = group['Crop_Variety'].unique()

    if len(unique_crop_varieties) < 2:
        continue  # Skip if there are fewer than 2 Crop_Varieties in this group

    for i in range(len(unique_crop_varieties)):
        for j in range(i + 1, len(unique_crop_varieties)):
            crop_variety_head = unique_crop_varieties[i]
            crop_variety_other = unique_crop_varieties[j]

            head_data = group[group['Crop_Variety'] == crop_variety_head]
            other_data = group[group['Crop_Variety'] == crop_variety_other]

            diff = head_data['GRAIN_YLD_13PCT'].values[0] - other_data['GRAIN_YLD_13PCT'].values[0]

            win = int(diff > 0)
            lose = int(diff < 0)
            draw = int(diff == 0)


            complete_dataset['CROP_SEASON'].append(group['CROP_SEASON'].values[0])
            complete_dataset['MACRO_REGION'].append(group['MACRO_REGION'].values[0])
            complete_dataset['Country'].append(group['Country'].values[0])
            complete_dataset['TRIAL_NAME'].append(trial_name)
            complete_dataset['REP_NO'].append(rep_no)
            complete_dataset['Crop_Variety_head'].append(crop_variety_head)
            complete_dataset['Crop_Variety_other'].append(crop_variety_other)
            complete_dataset['GRAIN_YLD_13PCT_head'].append(head_data['GRAIN_YLD_13PCT'].values[0])
            complete_dataset['GRAIN_YLD_13PCT_other'].append(other_data['GRAIN_YLD_13PCT'].values[0])
            complete_dataset['Difference'].append(diff)
            complete_dataset['Win'].append(win)
            complete_dataset['Lose'].append(lose)
            complete_dataset['Draw'].append(draw)

# Create a DataFrame from the comparison data
complete_dataset = pd.DataFrame(complete_dataset)

## Macro Region

In [8]:


# Group by Macro region to calculate means and sums
calc_mr = complete_dataset.groupby(['MACRO_REGION','Crop_Variety_head', 'Crop_Variety_other']).agg(
    Head_mean=('GRAIN_YLD_13PCT_head', 'mean'),
    Other_mean=('GRAIN_YLD_13PCT_other', 'mean'),
    difference_mean=('Difference', 'mean'),
    Total_wins=('Win', 'sum'),
    Total_comp=('Win', 'count')
).reset_index()

calc_mr['diff_perc'] = calc_mr['difference_mean'] / calc_mr['Other_mean']
calc_mr['wins_perc'] = calc_mr['Total_wins'] / calc_mr['Total_comp']


## Country

In [9]:
# Group by Macro region to calculate means and sums
calc_ct = complete_dataset.groupby(['Country','Crop_Variety_head', 'Crop_Variety_other']).agg(
    Head_mean=('GRAIN_YLD_13PCT_head', 'mean'),
    Other_mean=('GRAIN_YLD_13PCT_other', 'mean'),
    difference_mean=('Difference', 'mean'),
    Total_wins=('Win', 'sum'),
    Total_comp=('Win', 'count')
).reset_index()

calc_ct['diff_perc'] = calc_ct['difference_mean'] / calc_ct['Other_mean']
calc_ct['wins_perc'] = calc_ct['Total_wins'] / calc_ct['Total_comp']


## Trial

In [10]:
# Group by Macro region to calculate means and sums
calc_trial = complete_dataset.groupby(['TRIAL_NAME','Crop_Variety_head', 'Crop_Variety_other']).agg(
    Head_mean=('GRAIN_YLD_13PCT_head', 'mean'),
    Other_mean=('GRAIN_YLD_13PCT_other', 'mean'),
    difference_mean=('Difference', 'mean'),
    Total_wins=('Win', 'sum'),
    Total_comp=('Win', 'count')
).reset_index()

calc_trial['diff_perc'] = calc_trial['difference_mean'] / calc_trial['Other_mean']
calc_trial['wins_perc'] = calc_trial['Total_wins'] / calc_trial['Total_comp']


## General

In [11]:
# Group by Macro region to calculate means and sums
calc_tt = complete_dataset.groupby(['Crop_Variety_head', 'Crop_Variety_other']).agg(
    Head_mean=('GRAIN_YLD_13PCT_head', 'mean'),
    Other_mean=('GRAIN_YLD_13PCT_other', 'mean'),
    difference_mean=('Difference', 'mean'),
    Total_wins=('Win', 'sum'),
    Total_comp=('Win', 'count')
).reset_index()

calc_tt['diff_perc'] = calc_tt['difference_mean'] / calc_tt['Other_mean']
calc_tt['wins_perc'] = calc_tt['Total_wins'] / calc_tt['Total_comp']


## Downloads

In [15]:
final_df.to_csv("clean_df.csv")
calc_mr.to_csv("calculated_mr.csv")
calc_ct.to_csv("calculated_country.csv")
calc_trial.to_csv("calculated_trial.csv")
calc_tt.to_csv("calculated_total.csv")